# Notebook 05 — Drift Detection

## Purpose
Detect data drift and performance decay in PropertyIQ models post-deployment.
Run formal statistical tests (KS, Chi-Square) on 34,645 properties evaluated after training.
Compute rolling MAPE windows to quantify accuracy degradation over time.
Analyze per-city shifts to identify regional retraining needs.
Generate confidence scores for drift window properties.
Save comprehensive drift metadata for dashboard consumption.

## Why Drift Detection Matters

**Data Drift** — Input features have shifted
> "The properties being evaluated today look different from the properties the model was trained on."
> Example: A model trained 2017-2019 (pre-COVID) evaluates 2023 properties (post-COVID).
> Mumbai property prices doubled. Model sees different feature distributions.
> Without drift detection, model makes predictions based on outdated patterns.

**Performance Drift** — Model accuracy has degraded
> "The model was 13.8% MAPE on validation, but on recent data it may be 20%+ MAPE."
> Bank approves loans at 40% below actual market value → massive credit losses.
> Rolling MAPE windows detect when model stops working in production.

## Inputs
- `data/processed/features_train.csv` — 69,353 rows × 15 cols (baseline training set)
- `data/processed/features_drift.csv` — 34,645 rows × 15 cols (drift window: post-deployment data)
- `data/processed/features_val.csv` — 11,558 rows × 15 cols (validation reference)
- `models/sale_price_v1.pkl` — Trained RandomForest (13.84% MAPE on validation)
- `outputs/model_registry.json` — Model metadata (MAPE, features, hyperparams)

## Outputs
- `outputs/drift_results.json` — Comprehensive drift metadata (all test results, per-city analysis, confidence distribution)
- `outputs/ks_results.json` — KS-test results by feature (dashboard reads this)
- `outputs/rolling_mape.json` — Rolling MAPE windows array (dashboard reads this)
- `outputs/drift_analysis.png` — 2×2 diagnostic plots (KS stats, rolling MAPE, per-city, distribution shift)

## Drift Detection Thresholds
- **KS_THRESHOLD** = 0.10 — KS statistic above = data drift detected
- **P_VALUE_THRESHOLD** = 0.05 — p-value below = statistically significant
- **CHI2_P_THRESHOLD** = 0.05 — categorical feature drift threshold
- **MAPE_DRIFT_THRESHOLD** = 20.0% — Performance drift alert (vs. 13.84% baseline)
- **ROLLING_WINDOW_SIZE** = 500 rows per window

## Features Tested

**Continuous (11):** total_sqft, bhk, bath, bath_per_bhk, sqft_per_bhk, city_median_price_sqft, locality_median_price_sqft, rbi_hpi_avg, hpi_tier_interaction, sqft_city_interaction, bhk_sqft_ratio

**Categorical (2):** city_tier_encoded, is_large_property

In [ ]:
# ── CELL 2 ─ Imports & Constants ──────────────

from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import joblib
import json
import logging
import time
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Define base paths using absolute reference
BASE_DIR = Path(__file__).resolve().parent.parent
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"

# Drift detection thresholds
KS_THRESHOLD = 0.10
P_VALUE_THRESHOLD = 0.05
CHI2_P_THRESHOLD = 0.05
MAPE_DRIFT_THRESHOLD = 20.0
ROLLING_WINDOW_SIZE = 500

# Confidence score thresholds
CONFIDENCE_THRESHOLD_TRUSTED = 75.0
CONFIDENCE_THRESHOLD_CAUTION = 50.0

# Feature definitions
CONTINUOUS_FEATURES = [
    'total_sqft',
    'bhk',
    'bath',
    'bath_per_bhk',
    'sqft_per_bhk',
    'city_median_price_sqft',
    'locality_median_price_sqft',
    'rbi_hpi_avg',
    'hpi_tier_interaction',
    'sqft_city_interaction',
    'bhk_sqft_ratio'
]

CATEGORICAL_FEATURES = [
    'city_tier_encoded',
    'is_large_property'
]

FEATURE_COLS = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES

TARGET_SALE = 'price_sqft'

# Cities in dataset
CITIES = [
    'Mumbai', 'Bengaluru', 'Pune', 'Delhi',
    'Chennai', 'Kolkata', 'Hyderabad'
]

logger.info("Notebook 05: Drift Detection initialized")
logger.info(f"Continuous features: {len(CONTINUOUS_FEATURES)}")
logger.info(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
logger.info(f"Drift thresholds: KS={KS_THRESHOLD}, MAPE={MAPE_DRIFT_THRESHOLD}%")

In [ ]:
# ── CELL 3 ─ Load Data & Model ───────────────
# Load baseline training, drift window, and reference validation datasets
# Load trained model and metadata registry

logger.info("Loading datasets and trained model...")

try:
    # Load feature datasets
    df_train = pd.read_csv(DATA_PROCESSED / "features_train.csv")
    df_drift = pd.read_csv(DATA_PROCESSED / "features_drift.csv")
    df_val = pd.read_csv(DATA_PROCESSED / "features_val.csv")
    
    logger.info(f"✓ Loaded features_train.csv: {df_train.shape}")
    logger.info(f"✓ Loaded features_drift.csv: {df_drift.shape}")
    logger.info(f"✓ Loaded features_val.csv: {df_val.shape}")
    
    # Load trained model
    sale_model = joblib.load(MODELS_DIR / "sale_price_v1.pkl")
    logger.info(f"✓ Loaded sale_price_v1.pkl")
    
    # Load model metadata
    with open(OUTPUTS_DIR / "model_registry.json", 'r') as f:
        model_registry = json.load(f)
    
    base_mape = model_registry['sale_price_v1']['val_mape_percent']
    logger.info(f"✓ Loaded model_registry.json (base MAPE: {base_mape}%)")
    
    # Extract feature matrices and fill NaN with 0
    X_train = df_train[FEATURE_COLS].fillna(0).copy()
    y_train = df_train[TARGET_SALE].copy()
    
    X_drift = df_drift[FEATURE_COLS].fillna(0).copy()
    y_drift = df_drift[TARGET_SALE].copy()
    
    X_val = df_val[FEATURE_COLS].fillna(0).copy()
    y_val = df_val[TARGET_SALE].copy()
    
    logger.info(f"✓ Extracted and filled features: NaN → 0")
    
    # Critical assertions
    assert X_train.shape[1] == 13, f"X_train has {X_train.shape[1]} features, expected 13"
    assert X_drift.shape[1] == 13, f"X_drift has {X_drift.shape[1]} features, expected 13"
    assert X_val.shape[1] == 13, f"X_val has {X_val.shape[1]} features, expected 13"
    
    assert X_train.isna().sum().sum() == 0, "NaNs remain in X_train"
    assert X_drift.isna().sum().sum() == 0, "NaNs remain in X_drift"
    assert X_val.isna().sum().sum() == 0, "NaNs remain in X_val"
    
    assert len(df_train) == 69353, f"Train size {len(df_train)}, expected 69353"
    assert len(df_drift) == 34645, f"Drift size {len(df_drift)}, expected 34645"
    assert len(df_val) == 11558, f"Val size {len(df_val)}, expected 11558"
    
    logger.info("✓ All assertions passed: shapes, NaNs, row counts")
    
    # Print dataset summary
    print("\n" + "╔" + "═"*68 + "╗")
    print("║" + " DRIFT DETECTION — INPUT DATASETS".center(68) + "║")
    print("╠" + "═"*68 + "╣")
    print(f"║ Baseline Training Set".ljust(69) + "║")
    print(f"║   Rows: {len(df_train):>10,}   Columns: {X_train.shape[1]:>2d}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Drift Window (Test Dataset)".ljust(69) + "║")
    print(f"║   Rows: {len(df_drift):>10,}   Columns: {X_drift.shape[1]:>2d}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Validation Reference Set".ljust(69) + "║")
    print(f"║   Rows: {len(df_val):>10,}   Columns: {X_val.shape[1]:>2d}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Trained Model".ljust(69) + "║")
    print(f"║   Algorithm : RandomForestRegressor (300 trees)".ljust(69) + "║")
    print(f"║   Base MAPE : {base_mape:>10.2f}%".ljust(69) + "║")
    print(f"║   OOB R²    : {model_registry['sale_price_v1']['oob_r2']:>10.4f}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Drift Thresholds".ljust(69) + "║")
    print(f"║   KS-test (continuous)  : {KS_THRESHOLD:>10.2f}".ljust(69) + "║")
    print(f"║   Chi-Square (categor.) : p < {P_VALUE_THRESHOLD:>9.2f}".ljust(69) + "║")
    print(f"║   MAPE drift alert      : > {MAPE_DRIFT_THRESHOLD:>9.1f}%".ljust(69) + "║")
    print("╚" + "═"*68 + "╝\n")
    
except Exception as e:
    logger.error(f"Error loading datasets and model: {e}")
    raise

In [ ]:
# ── CELL 4 ─ KS-Test Continuous Features ─────
# Kolmogorov-Smirnov test: compare train vs drift distributions
# Tests if continuous feature distributions have shifted

logger.info("Running KS-tests on continuous features...")

def run_ks_tests(train_df: pd.DataFrame,
                 drift_df: pd.DataFrame,
                 features: list,
                 ks_threshold: float,
                 p_threshold: float) -> dict:
    """
    Runs Kolmogorov-Smirnov test on each continuous feature.
    Compares training distribution vs drift distribution.
    
    KS statistic ranges 0-1:
      0 = distributions identical
      1 = completely different
    
    p-value: probability both distributions are from same source
      p < 0.05 = statistically significant difference (drift detected)
    
    Args:
        train_df: Baseline training features
        drift_df: Post-deployment drift window features
        features: List of continuous feature names to test
        ks_threshold: KS stat above this = flag drift
        p_threshold: p-value below this = flag drift
    
    Returns:
        dict mapping feature_name → {
            ks_stat: float (0-1),
            p_value: float (0-1),
            drift_detected: bool,
            severity: str ("HIGH", "MEDIUM", "LOW")
        }
    
    Example:
        >>> results = run_ks_tests(X_train, X_drift,
        ...     CONTINUOUS_FEATURES, 0.10, 0.05)
        >>> results['total_sqft']
        {'ks_stat': 0.187, 'p_value': 0.0000,
         'drift_detected': True, 'severity': 'HIGH'}
    """
    results = {}
    
    for feature in features:
        try:
            # Extract non-null values from both datasets
            train_vals = train_df[feature].dropna().values
            drift_vals = drift_df[feature].dropna().values
            
            # Run KS test
            ks_stat, p_value = stats.ks_2samp(train_vals, drift_vals)
            
            # Determine if drift detected
            drift_detected = (ks_stat > ks_threshold and p_value < p_threshold)
            
            # Assign severity based on KS statistic magnitude
            if ks_stat > 0.30:
                severity = "HIGH"
            elif ks_stat > 0.15:
                severity = "MEDIUM"
            else:
                severity = "LOW"
            
            results[feature] = {
                "ks_stat": float(ks_stat),
                "p_value": float(p_value),
                "drift_detected": drift_detected,
                "severity": severity
            }
            
        except Exception as e:
            logger.warning(f"Error testing feature {feature}: {e}")
            results[feature] = {
                "ks_stat": np.nan,
                "p_value": np.nan,
                "drift_detected": False,
                "severity": "ERROR"
            }
    
    return results


# Run KS tests
try:
    ks_results = run_ks_tests(
        X_train, X_drift,
        CONTINUOUS_FEATURES,
        KS_THRESHOLD,
        P_VALUE_THRESHOLD
    )
    
    logger.info(f"✓ KS-tests completed on {len(ks_results)} features")
    
    # Identify features with drift
    drifted_features = [
        f for f, r in ks_results.items()
        if r['drift_detected']
    ]
    
    # Find most drifted feature
    most_drifted = max(
        ks_results.items(),
        key=lambda x: x[1]['ks_stat']
    )
    
    logger.info(
        f"Features with drift: {len(drifted_features)} of {len(CONTINUOUS_FEATURES)}"
    )
    logger.info(f"Most drifted: {most_drifted[0]} (KS={most_drifted[1]['ks_stat']:.4f})")
    
    # Print KS test results table
    print("\n" + "═"*85)
    print("KS-TEST RESULTS — CONTINUOUS FEATURES")
    print("Baseline: training set (69,353 rows)")
    print("Test:     drift window (34,645 rows)")
    print("═"*85)
    print(f"{'Feature':<30} {'KS Stat':>10} {'p-value':>12} {'Drift?':>10} {'Severity':>10}")
    print("─"*85)
    
    # Sort by KS stat descending
    sorted_results = sorted(
        ks_results.items(),
        key=lambda x: x[1]['ks_stat'],
        reverse=True
    )
    
    for feature, result in sorted_results:
        drift_flag = "YES ⚠" if result['drift_detected'] else "NO"
        p_val_str = f"{result['p_value']:.4e}"
        
        print(
            f"{feature:<30} {result['ks_stat']:>10.4f} {p_val_str:>12} "
            f"{drift_flag:>10} {result['severity']:>10}"
        )
    
    print("═"*85)
    print(f"Features with drift: {len(drifted_features)} of {len(CONTINUOUS_FEATURES)}")
    print(f"Most drifted feature: {most_drifted[0]} (KS={most_drifted[1]['ks_stat']:.4f})")
    print(f"Overall severity: {'HIGH' if len(drifted_features) > 5 else 'MEDIUM' if len(drifted_features) > 2 else 'LOW'}")
    print("═"*85 + "\n")
    
    assert len(ks_results) == len(CONTINUOUS_FEATURES), "KS test incomplete"
    logger.info("✓ KS-test results assertions passed")
    
except Exception as e:
    logger.error(f"Error running KS tests: {e}")
    raise

In [ ]:
# ── CELL 5 ─ Chi-Square Categorical Features ──
# Test if categorical feature distributions have shifted

logger.info("Running Chi-Square tests on categorical features...")

def run_chi2_tests(train_df: pd.DataFrame,
                   drift_df: pd.DataFrame,
                   features: list,
                   p_threshold: float) -> dict:
    """
    Runs Chi-Square test on each categorical feature.
    Tests if distribution of categories has shifted between train and drift.
    
    Chi-Square statistic: measures difference between observed and expected counts
      Higher value = larger difference
    
    p-value: probability both have same category distribution
      p < 0.05 = significant difference (drift detected)
    
    Args:
        train_df: Baseline training features
        drift_df: Post-deployment drift window features
        features: List of categorical feature names
        p_threshold: p-value below this = drift detected
    
    Returns:
        dict mapping feature_name → {
            chi2_stat: float,
            p_value: float,
            drift_detected: bool,
            train_dist: dict (category -> pct),
            drift_dist: dict (category -> pct)
        }
    
    Example:
        >>> results = run_chi2_tests(X_train, X_drift,
        ...     CATEGORICAL_FEATURES, 0.05)
    """
    results = {}
    
    for feature in features:
        try:
            # Get value counts and normalize to percentages
            train_counts = train_df[feature].value_counts().sort_index()
            drift_counts = drift_df[feature].value_counts().sort_index()
            
            # Align categories (add 0 for missing)
            all_categories = sorted(set(train_counts.index) | set(drift_counts.index))
            
            train_counts = train_counts.reindex(all_categories, fill_value=0)
            drift_counts = drift_counts.reindex(all_categories, fill_value=0)
            
            # Convert to observed frequencies for Chi-Square
            # Expected = train distribution applied to drift sample size
            total_drift = drift_counts.sum()
            expected_counts = train_counts.sum() / train_counts.sum() * total_drift
            
            # Run Chi-Square test
            chi2_stat, p_value = stats.chisquare(
                drift_counts.values,
                f_exp=expected_counts.values
            )
            
            # Compute distributions for display
            train_dist = (train_counts / train_counts.sum() * 100).to_dict()
            drift_dist = (drift_counts / drift_counts.sum() * 100).to_dict()
            
            drift_detected = p_value < p_threshold
            
            results[feature] = {
                "chi2_stat": float(chi2_stat),
                "p_value": float(p_value),
                "drift_detected": drift_detected,
                "train_dist": {str(k): float(v) for k, v in train_dist.items()},
                "drift_dist": {str(k): float(v) for k, v in drift_dist.items()}
            }
            
        except Exception as e:
            logger.warning(f"Error testing categorical feature {feature}: {e}")
            results[feature] = {
                "chi2_stat": np.nan,
                "p_value": np.nan,
                "drift_detected": False,
                "train_dist": {},
                "drift_dist": {}
            }
    
    return results


# Run Chi-Square tests
try:
    chi2_results = run_chi2_tests(
        X_train, X_drift,
        CATEGORICAL_FEATURES,
        CHI2_P_THRESHOLD
    )
    
    logger.info(f"✓ Chi-Square tests completed on {len(chi2_results)} features")
    
    # Print Chi-Square results
    print("\n" + "═"*85)
    print("CHI-SQUARE TEST RESULTS — CATEGORICAL FEATURES")
    print("Tests if category distributions have shifted")
    print("═"*85)
    
    for feature, result in chi2_results.items():
        drift_flag = "YES ⚠" if result['drift_detected'] else "NO"
        
        print(f"\n{feature}")
        print(f"  Chi² Statistic: {result['chi2_stat']:.4f}   p-value: {result['p_value']:.4f}   Drift: {drift_flag}")
        
        # Print distribution comparison
        print(f"  Distribution comparison:")
        
        all_cats = sorted(
            set(result['train_dist'].keys()) | set(result['drift_dist'].keys())
        )
        
        for cat in all_cats:
            train_pct = result['train_dist'].get(cat, 0.0)
            drift_pct = result['drift_dist'].get(cat, 0.0)
            change = drift_pct - train_pct
            arrow = "↑" if change > 2 else "↓" if change < -2 else "→"
            
            print(f"    Category {cat}: Train {train_pct:6.1f}%  →  Drift {drift_pct:6.1f}%  ({arrow} {abs(change):5.1f}%)")
    
    print("\n" + "═"*85 + "\n")
    
    assert len(chi2_results) == len(CATEGORICAL_FEATURES), "Chi-Square test incomplete"
    logger.info("✓ Chi-Square test results assertions passed")
    
except Exception as e:
    logger.error(f"Error running Chi-Square tests: {e}")
    raise

In [ ]:
# ── CELL 6 ─ Rolling MAPE Performance Drift ───
# Compute MAPE in rolling windows to detect performance decay over drift window
# Row order treated as temporal proxy: early rows = lower prices (pre-COVID),
# later rows = higher prices (post-COVID)

logger.info("Computing rolling MAPE windows...")

def compute_rolling_mape(model,
                         df: pd.DataFrame,
                         feature_cols: list,
                         target_col: str,
                         window_size: int,
                         base_mape: float) -> pd.DataFrame:
    """
    Computes MAPE in rolling windows across drift dataset.
    Detects performance decay: if MAPE rises above threshold,
    model accuracy is degrading.
    
    Row order is temporal proxy:
      Early rows (lower price_sqft) ≈ pre-COVID properties
      Later rows (higher price_sqft) ≈ post-COVID properties
    
    Args:
        model: Trained RandomForestRegressor
        df: Drift window DataFrame (sorted by price_sqft ascending)
        feature_cols: List of feature column names
        target_col: Target column name ('price_sqft')
        window_size: Rows per rolling window (500)
        base_mape: Baseline validation MAPE for comparison
    
    Returns:
        DataFrame with columns:
          window_idx: Window number (1-based)
          start_row: Starting row index
          end_row: Ending row index
          window_mape: MAPE % for this window
          vs_baseline: window_mape - base_mape
          alert: bool, True if window_mape > MAPE_DRIFT_THRESHOLD
    
    Example:
        >>> rolling = compute_rolling_mape(
        ...     sale_model, df_drift,
        ...     FEATURE_COLS, 'price_sqft',
        ...     500, 13.84)
        >>> rolling[rolling['alert']].shape[0]  # windows above threshold
        3
    """
    # Ensure drift dataframe is sorted by price_sqft (row order = time proxy)
    df_sorted = df.sort_values(target_col).reset_index(drop=True)
    
    rolling_results = []
    
    n_rows = len(df_sorted)
    n_windows = (n_rows + window_size - 1) // window_size
    
    for window_idx in range(n_windows):
        start_row = window_idx * window_size
        end_row = min(start_row + window_size, n_rows)
        
        # Extract window data
        window_df = df_sorted.iloc[start_row:end_row]
        
        X_window = window_df[feature_cols].fillna(0).values
        y_window = window_df[target_col].values
        
        # Skip if window too small
        if len(X_window) < 10:
            continue
        
        # Generate predictions
        y_pred_window = model.predict(X_window)
        
        # Compute MAPE
        window_mape = mean_absolute_percentage_error(y_window, y_pred_window) * 100
        
        # Compare to baseline
        vs_baseline = window_mape - base_mape
        
        # Flag alert
        alert = window_mape > MAPE_DRIFT_THRESHOLD
        
        rolling_results.append({
            'window_idx': window_idx + 1,
            'start_row': start_row,
            'end_row': end_row,
            'window_rows': end_row - start_row,
            'window_mape': round(window_mape, 2),
            'vs_baseline': round(vs_baseline, 2),
            'alert': alert
        })
    
    return pd.DataFrame(rolling_results)


# Compute rolling MAPE
try:
    rolling_mape_df = compute_rolling_mape(
        sale_model,
        df_drift,
        FEATURE_COLS,
        TARGET_SALE,
        ROLLING_WINDOW_SIZE,
        base_mape
    )
    
    logger.info(f"✓ Rolling MAPE computed: {len(rolling_mape_df)} windows")
    
    # Calculate summary metrics
    windows_above_threshold = rolling_mape_df['alert'].sum()
    peak_mape = rolling_mape_df['window_mape'].max()
    peak_window_idx = rolling_mape_df.loc[rolling_mape_df['window_mape'].idxmax(), 'window_idx']
    
    logger.info(f"Windows above {MAPE_DRIFT_THRESHOLD}%: {windows_above_threshold}")
    logger.info(f"Peak MAPE: {peak_mape}% (window {peak_window_idx})")
    
    # Print rolling MAPE table
    print("\n" + "═"*95)
    print("ROLLING MAPE — PERFORMANCE DRIFT ANALYSIS")
    print(f"Base MAPE (validation): {base_mape:.2f}%")
    print(f"Alert threshold:        {MAPE_DRIFT_THRESHOLD:.1f}%")
    print(f"Window size:            {ROLLING_WINDOW_SIZE} rows")
    print("═"*95)
    print(f"{'Window':<8} {'Rows':>10} {'Start-End':>15} {'MAPE':>8} {'vs Base':>10} {'Alert':>10}")
    print("─"*95)
    
    for _, row in rolling_mape_df.iterrows():
        alert_flag = "YES ⚠" if row['alert'] else "NO"
        rows_str = f"{row['start_row']}-{row['end_row']}"
        
        print(
            f"W-{int(row['window_idx']):02d}    "
            f"{int(row['window_rows']):>10}    {rows_str:>15}    "
            f"{row['window_mape']:>7.1f}%   {row['vs_baseline']:>+9.1f}%   {alert_flag:>10}"
        )
    
    print("═"*95)
    print(f"Windows above {MAPE_DRIFT_THRESHOLD}%: {windows_above_threshold} of {len(rolling_mape_df)}")
    if windows_above_threshold > 0:
        print(f"⚠ PERFORMANCE DRIFT DETECTED")
    else:
        print(f"✓ No windows exceed {MAPE_DRIFT_THRESHOLD}% threshold")
    print(f"Peak MAPE: {peak_mape:.1f}% (Window {int(peak_window_idx)})")
    print("═"*95 + "\n")
    
    assert len(rolling_mape_df) > 0, "Rolling MAPE computation failed"
    logger.info("✓ Rolling MAPE assertions passed")
    
except Exception as e:
    logger.error(f"Error computing rolling MAPE: {e}")
    raise

In [ ]:
# ── CELL 7 ─ Per-City Drift Analysis ──────────
# Analyze drift separately for each city
# Some cities drift more than others due to local market conditions

logger.info("Analyzing per-city drift...")

def analyze_city_drift(train_df: pd.DataFrame,
                       drift_df: pd.DataFrame,
                       model,
                       feature_cols: list) -> dict:
    """
    Runs drift analysis separately per city.
    
    Some cities experience larger price shifts due to local conditions:
      Mumbai: prices doubled post-COVID (+100-200%)
      Kolkata: prices changed less (+20-40%)
    
    City-level drift informs retraining strategy:
      HIGH-drift cities need urgent model updates
      LOW-drift cities can wait
    
    Args:
        train_df: Training data with city column
        drift_df: Drift data with city column
        model: Trained sale price model
        feature_cols: Feature column names
    
    Returns:
        dict mapping city_name → {
            ks_stat: float,
            p_value: float,
            train_mean_price: float,
            drift_mean_price: float,
            price_shift_pct: float,
            drift_mape: float
        }
    
    Example:
        >>> city_drift = analyze_city_drift(
        ...     df_train, df_drift, model, FEATURE_COLS)
        >>> city_drift['Mumbai']
        {'ks_stat': 0.287, 'p_value': 0.0,
         'train_mean_price': ..., 'drift_mean_price': ...,
         'price_shift_pct': 155.9, 'drift_mape': 18.5}
    """
    city_results = {}
    
    for city in sorted(train_df['city'].unique()):
        try:
            # Filter by city
            train_city = train_df[train_df['city'] == city]
            drift_city = drift_df[drift_df['city'] == city]
            
            if len(train_city) < 50 or len(drift_city) < 50:
                # Skip cities with insufficient data
                continue
            
            # KS test on price distributions
            ks_stat, p_value = stats.ks_2samp(
                train_city[TARGET_SALE],
                drift_city[TARGET_SALE]
            )
            
            # Price shift analysis
            train_mean_price = train_city[TARGET_SALE].mean()
            drift_mean_price = drift_city[TARGET_SALE].mean()
            price_shift_pct = (drift_mean_price - train_mean_price) / train_mean_price * 100
            
            # MAPE on city subset
            X_city = drift_city[feature_cols].fillna(0).values
            y_city = drift_city[TARGET_SALE].values
            y_pred_city = model.predict(X_city)
            drift_mape_city = mean_absolute_percentage_error(y_city, y_pred_city) * 100
            
            city_results[city] = {
                'ks_stat': float(ks_stat),
                'p_value': float(p_value),
                'train_mean_price': float(train_mean_price),
                'drift_mean_price': float(drift_mean_price),
                'price_shift_pct': float(price_shift_pct),
                'drift_mape': float(drift_mape_city)
            }
            
        except Exception as e:
            logger.warning(f"Error analyzing city {city}: {e}")
            continue
    
    return city_results


# Run per-city analysis
try:
    city_drift = analyze_city_drift(
        df_train, df_drift,
        sale_model,
        FEATURE_COLS
    )
    
    logger.info(f"✓ Per-city drift analysis completed for {len(city_drift)} cities")
    
    # Sort by KS stat (most drifted first)
    city_drift_sorted = sorted(
        city_drift.items(),
        key=lambda x: x[1]['ks_stat'],
        reverse=True
    )
    
    most_drifted_city = city_drift_sorted[0][0] if city_drift_sorted else "N/A"
    least_drifted_city = city_drift_sorted[-1][0] if city_drift_sorted else "N/A"
    
    logger.info(f"Most drifted city: {most_drifted_city}")
    logger.info(f"Least drifted city: {least_drifted_city}")
    
    # Print per-city drift table
    print("\n" + "═"*100)
    print("PER-CITY DRIFT ANALYSIS")
    print("═"*100)
    print(f"{'City':<15} {'KS Stat':>10} {'p-value':>12} {'Price Shift':>15} {'Drift MAPE':>12}")
    print("─"*100)
    
    for city, result in city_drift_sorted:
        p_val_str = f"{result['p_value']:.4e}"
        shift_str = f"{result['price_shift_pct']:+.1f}%"
        alert = "⚠" if result['ks_stat'] > 0.2 or result['drift_mape'] > 18 else ""
        
        print(
            f"{city:<15} {result['ks_stat']:>10.4f} {p_val_str:>12} "
            f"{shift_str:>15} {result['drift_mape']:>11.1f}%  {alert}"
        )
    
    print("═"*100)
    print(f"Most drifted city:  {most_drifted_city} (KS={city_drift[most_drifted_city]['ks_stat']:.4f})")
    print(f"Least drifted city: {least_drifted_city} (KS={city_drift[least_drifted_city]['ks_stat']:.4f})")
    print("═"*100 + "\n")
    
    assert len(city_drift) > 0, "City drift analysis failed"
    logger.info("✓ Per-city drift assertions passed")
    
except Exception as e:
    logger.error(f"Error in per-city drift analysis: {e}")
    raise

In [ ]:
# ── CELL 8 ─ Drift Visualization ──────────────
# Create 2×2 diagnostic plot: KS stats, rolling MAPE, per-city, distribution shift

logger.info("Creating drift visualization...")

try:
    # Create 2×2 figure with gridspec
    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
    
    # ──── PLOT 1: KS Statistics by Feature (top-left) ────
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Sort KS results by magnitude
    ks_sorted = sorted(
        ks_results.items(),
        key=lambda x: x[1]['ks_stat'],
        reverse=True
    )
    
    features_ks = [f for f, _ in ks_sorted]
    stats_ks = [r['ks_stat'] for _, r in ks_sorted]
    colors_ks = ['#e74c3c' if s > KS_THRESHOLD else '#27ae60' for s in stats_ks]
    
    ax1.barh(features_ks, stats_ks, color=colors_ks, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax1.axvline(KS_THRESHOLD, color='orange', linestyle='--', linewidth=2.5, label=f'Threshold={KS_THRESHOLD}')
    ax1.set_xlabel('KS Statistic', fontsize=11, fontweight='bold')
    ax1.set_title('KS Statistics by Feature\n(Higher = More Drift)', fontsize=12, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    ax1.legend(fontsize=10)
    
    # ──── PLOT 2: Rolling MAPE by Window (top-right) ────
    ax2 = fig.add_subplot(gs[0, 1])
    
    windows = rolling_mape_df['window_idx'].values
    mapes = rolling_mape_df['window_mape'].values
    
    # Color windows by alert status
    colors_mape = ['#e74c3c' if m > MAPE_DRIFT_THRESHOLD else '#27ae60' for m in mapes]
    
    ax2.plot(windows, mapes, marker='o', linewidth=2, markersize=6, color='#2c3e50', label='Window MAPE')
    ax2.scatter(windows, mapes, c=colors_mape, s=100, alpha=0.7, edgecolor='black', linewidth=1.2)
    ax2.axhline(base_mape, color='#3498db', linestyle='--', linewidth=2.5, label=f'Base MAPE={base_mape:.1f}%')
    ax2.axhline(MAPE_DRIFT_THRESHOLD, color='#e74c3c', linestyle='--', linewidth=2.5,
                label=f'Alert Threshold={MAPE_DRIFT_THRESHOLD:.1f}%')
    ax2.fill_between(windows, MAPE_DRIFT_THRESHOLD, mapes.max() + 2, alpha=0.1, color='red', label='Alert zone')
    
    ax2.set_xlabel('Window Index', fontsize=11, fontweight='bold')
    ax2.set_ylabel('MAPE %', fontsize=11, fontweight='bold')
    ax2.set_title('Rolling MAPE — Performance Over Time\n(Row order = Temporal proxy)', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=10, loc='best')
    
    # ──── PLOT 3: Per-City KS Stats (bottom-left) ────
    ax3 = fig.add_subplot(gs[1, 0])
    
    cities_sorted = [c for c, _ in city_drift_sorted]
    ks_city = [city_drift[c]['ks_stat'] for c in cities_sorted]
    colors_city = ['#e74c3c' if ks > 0.2 else '#f39c12' if ks > 0.1 else '#27ae60' for ks in ks_city]
    
    ax3.barh(cities_sorted, ks_city, color=colors_city, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax3.set_xlabel('KS Statistic', fontsize=11, fontweight='bold')
    ax3.set_title('Drift Severity by City\n(Higher = More Drifted)', fontsize=12, fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
    
    # ──── PLOT 4: Distribution of Most Drifted Feature (bottom-right) ────
    ax4 = fig.add_subplot(gs[1, 1])
    
    most_drifted_feature = most_drifted[0]
    
    train_vals = X_train[most_drifted_feature].dropna().values
    drift_vals = X_drift[most_drifted_feature].dropna().values
    
    # Create overlapping histograms
    ax4.hist(train_vals, bins=40, alpha=0.6, label='Train (baseline)',
             color='#3498db', edgecolor='black', linewidth=0.5)
    ax4.hist(drift_vals, bins=40, alpha=0.6, label='Drift (test)',
             color='#e74c3c', edgecolor='black', linewidth=0.5)
    
    ax4.set_xlabel(f'{most_drifted_feature}', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax4.set_title(f'Distribution Shift — {most_drifted_feature}\n(Most drifted feature)', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Overall title
    plt.suptitle('Drift Detection Analysis — Comprehensive Diagnostic View',
                 fontsize=14, fontweight='bold', y=0.995)
    
    # Save figure
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    plot_path = OUTPUTS_DIR / "drift_analysis.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    logger.info(f"✓ Drift visualization saved: {plot_path}")
    
    plt.close()
    
except Exception as e:
    logger.error(f"Error creating drift visualization: {e}")
    raise

In [ ]:
# ── CELL 9 ─ Confidence Distribution ──────────
# Compute confidence scores for drift window sample
# Expected: most properties score CAUTION or DANGER (reflecting model uncertainty)

logger.info("Computing confidence scores for drift window sample...")

def compute_confidence_scores(model, X_input: np.ndarray, base_mape: float) -> np.ndarray:
    """
    Computes prediction confidence score 0-100 using tree variance.
    
    Method:
      1. Extract predictions from all trees in forest
      2. Compute coefficient of variation: CV = std / mean
      3. Convert to raw confidence: 1 - CV*5 (clipped 0-1)
      4. Apply MAPE penalty: mape_factor = base_mape / 100
      5. Final confidence = raw_conf * (1 - mape_factor*0.5) * 100
    
    Interpretation:
      >= 75: TRUSTED — safe to proceed
      50-75: CAUTION — verify before deciding
      < 50:  DANGER  — field verification required
    
    Args:
        model: Fitted RandomForestRegressor
        X_input: Feature array (n_samples × n_features)
        base_mape: Model base MAPE percentage
    
    Returns:
        np.array: Confidence scores 0-100 (one per input row)
    """
    # Get predictions from all individual trees
    tree_preds = np.array([
        tree.predict(X_input)
        for tree in model.estimators_
    ])  # Shape: (n_estimators, n_samples)
    
    # Compute mean and std of tree predictions
    mean_pred = tree_preds.mean(axis=0)
    std_pred = tree_preds.std(axis=0)
    
    # Coefficient of variation
    cv = std_pred / (np.abs(mean_pred) + 1e-9)
    
    # Convert CV to raw confidence (0-1)
    raw_conf = np.clip(1 - cv * 5, 0, 1)
    
    # Apply MAPE penalty
    mape_penalty = base_mape / 100
    final_conf = raw_conf * (1 - mape_penalty * 0.5)
    
    # Scale to 0-100
    return np.clip(final_conf * 100, 0, 100)


try:
    # Sample 1000 rows from drift window
    sample_size = min(1000, len(X_drift))
    sample_indices = np.random.choice(len(X_drift), sample_size, replace=False)
    
    X_sample = X_drift.iloc[sample_indices].values
    
    # Compute confidence scores
    confidence_scores = compute_confidence_scores(sale_model, X_sample, base_mape)
    
    logger.info(f"✓ Computed confidence scores for {len(confidence_scores)} properties")
    
    # Compute confidence distribution
    trusted_count = (confidence_scores >= CONFIDENCE_THRESHOLD_TRUSTED).sum()
    caution_count = ((confidence_scores >= CONFIDENCE_THRESHOLD_CAUTION) & 
                     (confidence_scores < CONFIDENCE_THRESHOLD_TRUSTED)).sum()
    danger_count = (confidence_scores < CONFIDENCE_THRESHOLD_CAUTION).sum()
    
    trusted_pct = trusted_count / len(confidence_scores) * 100
    caution_pct = caution_count / len(confidence_scores) * 100
    danger_pct = danger_count / len(confidence_scores) * 100
    
    mean_confidence = confidence_scores.mean()
    median_confidence = np.median(confidence_scores)
    
    logger.info(f"Confidence distribution: Trusted={trusted_pct:.1f}%, Caution={caution_pct:.1f}%, Danger={danger_pct:.1f}%")
    
    # Print confidence distribution
    print("\n" + "╔" + "═"*68 + "╗")
    print("║" + " CONFIDENCE SCORE DISTRIBUTION — DRIFT WINDOW".center(68) + "║")
    print("╠" + "═"*68 + "╣")
    print(f"║ Sample size: {len(confidence_scores):>10,} properties".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ TRUSTED (≥{CONFIDENCE_THRESHOLD_TRUSTED:.0f})   : {trusted_count:>6,} ({trusted_pct:>6.1f}%)".ljust(69) + "║")
    print(f"║   Safe to proceed with loan evaluation".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ CAUTION ({CONFIDENCE_THRESHOLD_CAUTION:.0f}-{CONFIDENCE_THRESHOLD_TRUSTED:.0f}) : {caution_count:>6,} ({caution_pct:>6.1f}%)".ljust(69) + "║")
    print(f"║   Verify before deciding on loan".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ DANGER (<{CONFIDENCE_THRESHOLD_CAUTION:.0f})   : {danger_count:>6,} ({danger_pct:>6.1f}%)".ljust(69) + "║")
    print(f"║   Field verification required".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Mean confidence   : {mean_confidence:>10.1f}%".ljust(69) + "║")
    print(f"║ Median confidence : {median_confidence:>10.1f}%".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print("║ ✓ Expected: Most drift properties score CAUTION/DANGER".ljust(69) + "║")
    print("║   (This is correct — model uncertainty under drift conditions)".ljust(69) + "║")
    print("╚" + "═"*68 + "╝\n")
    
    assert len(confidence_scores) == sample_size, "Confidence score computation failed"
    logger.info("✓ Confidence distribution assertions passed")
    
except Exception as e:
    logger.error(f"Error computing confidence scores: {e}")
    raise

In [ ]:
# ── CELL 10 ─ Save All Results ────────────────
# Create comprehensive drift_results.json and supporting JSON files for dashboard

logger.info("Saving all drift detection results...")

try:
    # Create timestamp
    timestamp = pd.Timestamp.now().isoformat()
    
    # Build comprehensive drift results dictionary
    drift_results = {
        "generated_at": timestamp,
        "baseline_dataset": "features_train.csv",
        "drift_dataset": "features_drift.csv",
        "baseline_rows": int(len(df_train)),
        "drift_rows": int(len(df_drift)),
        "base_mape": float(base_mape),
        
        # KS-test results on continuous features
        "ks_results": {
            feature: {
                "ks_stat": result['ks_stat'],
                "p_value": result['p_value'],
                "drift_detected": result['drift_detected'],
                "severity": result['severity']
            }
            for feature, result in ks_results.items()
        },
        
        # Chi-Square results on categorical features
        "chi2_results": {
            feature: {
                "chi2_stat": result['chi2_stat'],
                "p_value": result['p_value'],
                "drift_detected": result['drift_detected'],
                "train_dist": result['train_dist'],
                "drift_dist": result['drift_dist']
            }
            for feature, result in chi2_results.items()
        },
        
        # Rolling MAPE analysis
        "rolling_mape": {
            "window_size": ROLLING_WINDOW_SIZE,
            "base_mape": float(base_mape),
            "alert_threshold": float(MAPE_DRIFT_THRESHOLD),
            "windows": rolling_mape_df.to_dict(orient='records'),
            "windows_above_threshold": int(windows_above_threshold),
            "peak_mape": float(peak_mape),
            "peak_window": int(peak_window_idx)
        },
        
        # Per-city drift analysis
        "city_drift": {
            city: {
                "ks_stat": result['ks_stat'],
                "p_value": result['p_value'],
                "train_mean_price": result['train_mean_price'],
                "drift_mean_price": result['drift_mean_price'],
                "price_shift_pct": result['price_shift_pct'],
                "drift_mape": result['drift_mape']
            }
            for city, result in city_drift.items()
        },
        
        # Confidence distribution
        "confidence_distribution": {
            "trusted_count": int(trusted_count),
            "trusted_pct": float(trusted_pct),
            "caution_count": int(caution_count),
            "caution_pct": float(caution_pct),
            "danger_count": int(danger_count),
            "danger_pct": float(danger_pct),
            "mean_confidence": float(mean_confidence),
            "median_confidence": float(median_confidence),
            "sample_size": int(len(confidence_scores))
        },
        
        # Summary and recommendation
        "summary": {
            "features_with_data_drift": int(len(drifted_features)),
            "total_features_tested": int(len(CONTINUOUS_FEATURES)),
            "most_drifted_feature": most_drifted[0],
            "most_drifted_feature_ks": float(most_drifted[1]['ks_stat']),
            "categories_with_drift": int(sum(1 for r in chi2_results.values() if r['drift_detected'])),
            "rolling_windows_above_threshold": int(windows_above_threshold),
            "total_rolling_windows": int(len(rolling_mape_df)),
            "most_drifted_city": most_drifted_city,
            "least_drifted_city": least_drifted_city,
            "overall_drift_severity": (
                "HIGH" if len(drifted_features) > 7 or windows_above_threshold > 3 else
                "MEDIUM" if len(drifted_features) > 3 or windows_above_threshold > 0 else
                "LOW"
            ),
            "recommendation": (
                "URGENT: Retrain model on recent properties. Multiple features drifted significantly. "
                f"Performance degrading in windows {windows_above_threshold} windows exceed {MAPE_DRIFT_THRESHOLD}% MAPE. "
                f"Focus retraining on {most_drifted_city} (KS={city_drift[most_drifted_city]['ks_stat']:.3f}). "
                if windows_above_threshold > 2 else
                "RECOMMENDED: Collect more recent training data and retrain within 2-3 months. "
                f"{len(drifted_features)} features show significant drift. "
                f"Current model acceptable but degrading. City {most_drifted_city} drifted most. "
                if len(drifted_features) > 3 else
                "ACCEPTABLE: Monitor performance monthly. Some feature drift detected but within tolerance. "
                "Retraining recommended in 6 months or when MAPE exceeds 18%."
            )
        }
    }
    
    # Save main drift results
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    drift_results_path = OUTPUTS_DIR / "drift_results.json"
    
    with open(drift_results_path, 'w') as f:
        json.dump(drift_results, f, indent=2)
    
    logger.info(f"✓ Main results saved: {drift_results_path}")
    
    # Save KS results separately for dashboard
    ks_results_flat = {
        feature: {
            "ks_stat": result['ks_stat'],
            "p_value": result['p_value'],
            "drift_detected": result['drift_detected'],
            "severity": result['severity']
        }
        for feature, result in ks_results.items()
    }
    
    ks_results_path = OUTPUTS_DIR / "ks_results.json"
    with open(ks_results_path, 'w') as f:
        json.dump(ks_results_flat, f, indent=2)
    
    logger.info(f"✓ KS results saved: {ks_results_path}")
    
    # Save rolling MAPE separately for dashboard
    rolling_mape_export = {
        "base_mape": float(base_mape),
        "threshold": float(MAPE_DRIFT_THRESHOLD),
        "window_size": ROLLING_WINDOW_SIZE,
        "peak_mape": float(peak_mape),
        "peak_window": int(peak_window_idx),
        "windows_above_threshold": int(windows_above_threshold),
        "total_windows": int(len(rolling_mape_df)),
        "windows": rolling_mape_df.to_dict(orient='records')
    }
    
    rolling_mape_path = OUTPUTS_DIR / "rolling_mape.json"
    with open(rolling_mape_path, 'w') as f:
        json.dump(rolling_mape_export, f, indent=2)
    
    logger.info(f"✓ Rolling MAPE saved: {rolling_mape_path}")
    
    # Print final summary
    print("\n" + "╔" + "═"*70 + "╗")
    print("║" + " NOTEBOOK 05 — DRIFT DETECTION FINAL SUMMARY".center(70) + "║")
    print("╠" + "═"*70 + "╣")
    print("║" + " "*70 + "║")
    print("║ KS-TEST RESULTS (Continuous Features)".ljust(71) + "║")
    print(f"║   Features tested      : {len(CONTINUOUS_FEATURES):>6}".ljust(71) + "║")
    print(f"║   Features drifted     : {len(drifted_features):>6}".ljust(71) + "║")
    print(f"║   Most drifted         : {most_drifted[0]:>30}".ljust(71) + "║")
    print(f"║   KS stat (most)       : {most_drifted[1]['ks_stat']:>30.4f}".ljust(71) + "║")
    print("║" + " "*70 + "║")
    print("║ PERFORMANCE DRIFT (Rolling MAPE)".ljust(71) + "║")
    print(f"║   Base MAPE            : {base_mape:>30.2f}%".ljust(71) + "║")
    print(f"║   Peak window MAPE     : {peak_mape:>30.2f}%".ljust(71) + "║")
    print(f"║   Windows (>20%)       : {windows_above_threshold:>30} of {len(rolling_mape_df)}".ljust(71) + "║")
    if windows_above_threshold > 0:
        print("║   ⚠ PERFORMANCE DRIFT ALERT — Review peak window performance".ljust(71) + "║")
    print("║" + " "*70 + "║")
    print("║ CITY-LEVEL DRIFT".ljust(71) + "║")
    print(f"║   Most drifted city    : {most_drifted_city:>30}".ljust(71) + "║")
    print(f"║   KS stat (city)       : {city_drift[most_drifted_city]['ks_stat']:>30.4f}".ljust(71) + "║")
    print(f"║   Price shift          : {city_drift[most_drifted_city]['price_shift_pct']:>+29.1f}%".ljust(71) + "║")
    print(f"║   Least drifted city   : {least_drifted_city:>30}".ljust(71) + "║")
    print("║" + " "*70 + "║")
    print("║ CONFIDENCE DISTRIBUTION (Drift Sample)".ljust(71) + "║")
    print(f"║   Trusted (≥75%)       : {trusted_pct:>30.1f}%".ljust(71) + "║")
    print(f"║   Caution (50-75%)     : {caution_pct:>30.1f}%".ljust(71) + "║")
    print(f"║   Danger (<50%)        : {danger_pct:>30.1f}%".ljust(71) + "║")
    print(f"║   Mean confidence      : {mean_confidence:>30.1f}%".ljust(71) + "║")
    print("║" + " "*70 + "║")
    print("║ OVERALL SEVERITY".ljust(71) + "║")
    print(f"║   Status               : {drift_results['summary']['overall_drift_severity']:>30}".ljust(71) + "║")
    print(f"║   Recommendation       : {drift_results['summary']['recommendation'][:47]}...".ljust(71) + "║")
    print("║" + " "*70 + "║")
    print("║ FILES SAVED:".ljust(71) + "║")
    print("║   ✓ outputs/drift_results.json".ljust(71) + "║")
    print("║   ✓ outputs/ks_results.json".ljust(71) + "║")
    print("║   ✓ outputs/rolling_mape.json".ljust(71) + "║")
    print("║   ✓ outputs/drift_analysis.png".ljust(71) + "║")
    print("║" + " "*70 + "║")
    print("║ READY FOR NOTEBOOK 06 — DASHBOARD CONSUMPTION".ljust(71) + "║")
    print("╚" + "═"*70 + "╝\n")
    
    logger.info("="*70)
    logger.info("NOTEBOOK 05 COMPLETE — All drift detection results saved")
    logger.info(f"Overall drift severity: {drift_results['summary']['overall_drift_severity']}")
    logger.info(f"Recommendation: {drift_results['summary']['recommendation'][:80]}...")
    logger.info("="*70)
    
except Exception as e:
    logger.error(f"Error saving drift results: {e}")
    raise